# 2026 COMP90042 Project — Group 073
*Fact-checking: two-stage retrieval (BM25 → BERT cross-encoder) + claim classifier.*

# Readme

**Before running:**

1. From your local machine, push the working branch to GitHub:
   `git push -u origin vita/google-colab`
2. Add a GitHub Personal Access Token to Colab Secrets:
   sidebar 🔑 → **Add new secret** → name `GITHUB_PAT`, value = your fine-grained PAT with **Contents: Read** on the repo, allow notebook access.
3. On Google Drive at `/content/drive/MyDrive/COMP90042_2026/`, place the data:
   - Required: `data/evidence.json` (~1 GB), `data/train-claims.json`, `data/dev-claims.json`, `data/test-claims-unlabelled.json`
   - Optional (saves ~30 min): `cache/bm25_index/`, `cache/bm25_train_top200.json`

Cell 1.1 clones the code from GitHub to `/content/COMP90042_2026` (Colab's fast local SSD) and symlinks `data/`, `cache/`, `checkpoints/`, `outputs/` from Drive — so code is git-managed and data persists across sessions.

**To resume training after a Colab disconnect:** set `RESUME_FROM` in cell 2.1 to the last saved epoch checkpoint, e.g. `'checkpoints/cross_encoder_epoch1.pt'`.

**Pipeline:** BM25 first-stage retrieval (1.2M → 200 candidates) → BERT cross-encoder reranker (top-4) → claim classifier.


# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 1.1 · Setup — Sync code from GitHub, mount Drive, install packages

import os

# Stop JAX/TF from preallocating GPU memory. Colab pre-loads JAX which by
# default grabs ~75% of T4 VRAM, causing CUDA OOM when PyTorch then tries
# to allocate. These env vars MUST be set before any GPU library is imported.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import shutil
import subprocess
import sys

from google.colab import drive, userdata

drive.mount("/content/drive")

# -- EDIT IF NEEDED ----------------------------------------------------------
GITHUB_USER = "VitaChien"
REPO_NAME = "COMP90042_2026"
BRANCH = "vita/retriever"
DRIVE_DATA = "/content/drive/MyDrive/COMP90042_2026"  # data + checkpoints persist here
PROJECT_ROOT = "/content/COMP90042_2026"  # code clones to fast local SSD
# ----------------------------------------------------------------------------

# Pull latest code from GitHub (clone on first run, pull on subsequent runs)
pat = userdata.get("GITHUB_PAT")
repo_url = f"https://{pat}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
if not os.path.exists(PROJECT_ROOT):
    subprocess.check_call(["git", "clone", "-b", BRANCH, repo_url, PROJECT_ROOT])
else:
    subprocess.check_call(["git", "-C", PROJECT_ROOT, "pull"])

# Symlink data directories from Drive into the cloned repo so data/checkpoints
# persist across Colab sessions while code stays on the fast local SSD.
for sub in ("data", "cache", "checkpoints", "outputs"):
    src = f"{DRIVE_DATA}/{sub}"
    dst = f"{PROJECT_ROOT}/{sub}"
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        shutil.rmtree(dst)  # replace tracked .gitkeep dir with symlink
    if not os.path.islink(dst):
        os.symlink(src, dst)

# Install Python dependencies
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "transformers>=4.40",
        "bm25s[full]>=0.3",
        "scikit-learn>=1.3",
        "tqdm>=4.65",
    ]
)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# @title 1.2 · Verify data files exist
from pathlib import Path

root = Path(PROJECT_ROOT)
checks = [
    ("data/evidence.json", "required (~1 GB)"),
    ("data/train-claims.json", "required"),
    ("data/dev-claims.json", "required"),
    ("data/test-claims-unlabelled.json", "required"),
    ("cache/bm25_index", "optional — saves ~10 min"),
    ("cache/bm25_train_top200.json", "optional — saves ~20 min"),
]
missing_required = False
for rel, note in checks:
    p = root / rel
    if p.exists():
        size = f"{p.stat().st_size/1e6:.0f} MB" if p.is_file() else "dir"
        print(f"  OK  {rel:45s} {size:10s}  ({note})")
    else:
        icon = "XX" if "required" in note else "--"
        print(f'  {icon}  {rel:45s} {"MISSING":10s}  ({note})')
        if "required" in note:
            missing_required = True

if missing_required:
    raise FileNotFoundError("Required data files are missing. Upload them to Drive first.")

In [ ]:
# @title 1.3 · Build BM25 index — idempotent, skips if cache/bm25_index/ already exists
# Takes ~10 min on first run; subsequent runs print 'already exists' and return.
import importlib.util

spec = importlib.util.spec_from_file_location("build_bm25", f"{PROJECT_ROOT}/scripts/build_bm25.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 2.1 · Train cross-encoder (configure RESUME_FROM / EPOCHS, then run)
#
# After a Colab disconnect:
#   1. Re-run cell 1.1 (will git-pull latest code and re-mount Drive).
#   2. Set RESUME_FROM below to the last saved epoch checkpoint path.
#   3. Re-run this cell.

import importlib.util
import sys

RESUME_FROM = None  # e.g. "checkpoints/cross_encoder_epoch1.pt"  (None = start fresh)
EPOCHS = 4  # override cfg.ce_epochs=2: with hard_negatives_per_pos=8 (~37k pairs)
# the loss was still dropping at epoch 2 in earlier runs.

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "train_cross_encoder", f"{PROJECT_ROOT}/scripts/train_cross_encoder.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = ["train_cross_encoder.py"]
if RESUME_FROM:
    sys.argv += ["--resume", RESUME_FROM]
if EPOCHS is not None:
    sys.argv += ["--epochs", str(EPOCHS)]

mod.main()
print("\nDone. Final checkpoint: checkpoints/cross_encoder.pt")

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 3.1 · Run retriever-only inference on dev + test + train (BM25 top-200 -> CE rerank top-K)
# Loads BM25 index, cross-encoder, and the ~1GB evidence corpus ONCE,
# then loops over splits. Skips any split whose output file already exists,
# so a Colab disconnect mid-run can be resumed by simply re-running the cell.
# Order: dev -> test -> train (cheap first; train is ~1228 claims, slowest).
import importlib.util
import sys
from pathlib import Path

SPLITS = ["dev", "test", "train"]
TOP_K = 4
BM25_TOP_K = 200

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "run_inference", f"{PROJECT_ROOT}/scripts/run_inference.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

cfg = mod.Config()
split_to_path = {"train": cfg.train_path, "dev": cfg.dev_path, "test": cfg.test_path}

# Decide which splits actually need running before paying the load cost.
pending = []
for split in SPLITS:
    out = cfg.output_dir / f"{split}-retriever-only-k{TOP_K}-bm25{BM25_TOP_K}.json"
    if out.exists():
        print(f"[{split}] skip - {out.name} already exists")
    else:
        pending.append((split, out))

if not pending:
    print("\nAll three split outputs already present. Delete a file under outputs/ to force a rerun.")
else:
    bm25, ce_tok, ce_model, evidence, device = mod.load_retriever_components(cfg)
    print(f"\nLoaded components on device={device}; running {len(pending)} split(s)...")
    for split, out in pending:
        print(f"\n=== {split} -> {out.name} ===")
        mod.run_retriever_only(
            split_to_path[split], out, TOP_K, bm25_top_k=BM25_TOP_K,
            bm25=bm25, ce_tok=ce_tok, ce_model=ce_model, evidence=evidence, device=device,
        )
        if split in {"train", "dev"}:
            m = mod.evaluate_predictions(out, split_to_path[split])
            print(f"[{split}] F={m['evidence_f']:.4f}  A={m['claim_accuracy']:.4f}  HM={m['harmonic_mean']:.4f}")


In [ ]:
# @title 3.2 · Evaluate retriever-only on dev — Evidence F, Claim Accuracy, Harmonic Mean
import subprocess
import sys

SPLIT = "dev"
TOP_K = 4
BM25_TOP_K = 200

PRED_FILE = f"outputs/{SPLIT}-retriever-only-k{TOP_K}-bm25{BM25_TOP_K}.json"
GT_FILE = f"data/{SPLIT}-claims.json"

subprocess.check_call(
    [
        sys.executable,
        f"{PROJECT_ROOT}/eval.py",
        "--predictions", PRED_FILE,
        "--groundtruth", GT_FILE,
        "--verbose",
    ]
)


## Object Oriented Programming codes here

The OOP implementation lives in `src/` (importable modules) and `scripts/` (entry points):

| Module | Key class / function |
|--------|----------------------|
| `src/config.py` | `Config` dataclass — all hyperparameters and paths |
| `src/data_loader.py` | `Claim` dataclass, `load_claims`, `load_evidence` |
| `src/retriever_bm25.py` | `BM25Retriever` — first-stage lexical retrieval |
| `src/retriever_cross_enc.py` | `CrossEncoderHead(nn.Module)` — BERT reranker |
| `src/hard_negatives.py` | `build_training_pairs` — hard-negative mining |
| `src/evaluator.py` | `evaluate` — evidence F, claim accuracy, harmonic mean |
    